In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Set, Dict
from collections import defaultdict

RANDOM_SEED = 41
np.random.seed(RANDOM_SEED)

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs
from rdkit import RDLogger
import time as t 
import random as r

RDLogger.DisableLog('rdApp.*')
print(f"Imported at {t.ctime()}")

Imported at Wed Mar  4 12:36:44 2026


In [20]:
"""
The scaffold folder structre is:
"scaffold_cv_output/
    folds/
        fold_0/
            test/
                test.csv
            train/
                train.csv
            val/
                val.csv
    hold_out/
        hold_out.csv


Its based on a scaffold split of the original dataset.
original bace dataset is stored in "../bace.csv"



"""

scaffold_folder = "scaffold_cv_output/"

# function to read one fold of the scaffold split
def read_scaffold_fold(fold_num: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    fold_path = f"{scaffold_folder}folds/fold_{fold_num}/"
    train_df = pd.read_csv(f"{fold_path}train/train.csv")
    val_df = pd.read_csv(f"{fold_path}val/val.csv")
    test_df = pd.read_csv(f"{fold_path}test/test.csv")
    return train_df, val_df, test_df

# function to read the hold out set
def read_hold_out_set() -> pd.DataFrame:
    hold_out_df = pd.read_csv(f"{scaffold_folder}hold_out/hold_out.csv")
    return hold_out_df


def load_original_dataset() -> pd.DataFrame:
    original_df = pd.read_csv("../bace.csv")
    return original_df

def is_clean_smiles(smiles: str,make_generic=False) -> tuple[bool, Chem.Mol|None]:
    # A simple check to see if the SMILES string can be parsed by RDKit
    mol = Chem.MolFromSmiles(smiles, sanitize=True)
    try:
        scaf_mol = MurckoScaffold.GetScaffoldForMol(mol)
    except Exception:
        print(f"Cant make scaffold for: {smiles}")
        return False, None
    if mol is None:
        print(f"Invalid SMILES: {smiles}")
        return False, None
    if make_generic:
        try:
            scaf_mol = MurckoScaffold.MakeScaffoldGeneric(scaf_mol)
        except Chem.AtomValenceException:
            print(f"Valence error for: {smiles}")
            return False,None
        except Exception:
            return False,None

    if scaf_mol is None or scaf_mol.GetNumAtoms() == 0:
        print(f"Empty scaffold for: {smiles}")
        return False, None

    try:
        scaf_smiles = Chem.MolToSmiles(scaf_mol, isomericSmiles=False)
        return True, scaf_mol
    except Exception:
        print(f"Error converting scaffold to SMILES for: {smiles}")
        return False, None


def analyze_splits(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame, hold_out_df: pd.DataFrame) -> None:
    # Analyze the splits for scaffold overlap and distribution of target variable
    def get_scaffold_set(df: pd.DataFrame,set_name: str|None) -> Set[str]:
        scaffolds = set()
        cl = 0
        for smiles in df['smiles']:
            
            clean, scaf_mol = is_clean_smiles(smiles)
            if clean:
                cl += 1
                scaf_smiles = Chem.MolToSmiles(scaf_mol, isomericSmiles=False)
                scaffolds.add(scaf_smiles)
        print(80*"=")
        print(f"Analyzing {set_name} set:")
        print(f"Processed {len(df)} SMILES, found {len(scaffolds)} unique scaffolds.")
        print(f"Clean SMILES: {cl}/{len(df)}")
        print(80*"=")
        return scaffolds

    train_scaffolds = get_scaffold_set(train_df,"train")
    val_scaffolds = get_scaffold_set(val_df,"val")
    test_scaffolds = get_scaffold_set(test_df,"test")
    hold_out_scaffolds = get_scaffold_set(hold_out_df,"hold_out")

    print(f"Train scaffolds: {len(train_scaffolds)}")
    print(f"Val scaffolds: {len(val_scaffolds)}")
    print(f"Test scaffolds: {len(test_scaffolds)}")
    print(f"Hold-out scaffolds: {len(hold_out_scaffolds)}")

    # Check for scaffold overlap
    train_val_overlap = train_scaffolds.intersection(val_scaffolds)
    train_test_overlap = train_scaffolds.intersection(test_scaffolds)
    val_test_overlap = val_scaffolds.intersection(test_scaffolds)

    print(f"Train-Val scaffold overlap: {len(train_val_overlap)}")
    print(f"Train-Test scaffold overlap: {len(train_test_overlap)}")
    print(f"Val-Test scaffold overlap: {len(val_test_overlap)}")



In [21]:
train_df, val_df, test_df = read_scaffold_fold(0)
hold_out_df = read_hold_out_set()
analyze_splits(train_df, val_df, test_df, hold_out_df)

Analyzing train set:
Processed 940 SMILES, found 424 unique scaffolds.
Clean SMILES: 940/940
Analyzing val set:
Processed 104 SMILES, found 41 unique scaffolds.
Clean SMILES: 104/104
Analyzing test set:
Processed 262 SMILES, found 96 unique scaffolds.
Clean SMILES: 262/262
Analyzing hold_out set:
Processed 202 SMILES, found 109 unique scaffolds.
Clean SMILES: 202/202
Train scaffolds: 424
Val scaffolds: 41
Test scaffolds: 96
Hold-out scaffolds: 109
Train-Val scaffold overlap: 0
Train-Test scaffold overlap: 0
Val-Test scaffold overlap: 0


In [5]:
print(train_df.head())

                                              smiles     pIC50
0  S1(=O)(=O)C[C@@H](Cc2cc(O[C@H](COCC)C(F)(F)F)c...  8.698970
1  S1(=O)(=O)C[C@@H](Cc2cc(C[C@H](O)C(F)(F)F)c(N)...  7.309804
2  Brc1cc(cc(F)c1N)C[C@@H]1CS(=O)(=O)C[C@H]([NH2+...  5.863279
3  S1(=O)(=O)CC(Cc2cc(OC(C(F)(F)F)C(F)(F)F)c(N)c(...  8.698970
4  S1(=O)(=O)CC(Cc2cc(O[C@H](COC)C(F)(F)F)c(N)c(F...  8.698970
